Improved experiments (v4). One code cell. Needs GPU and Internet On. Use Save Version > Save & Run All (Commit). Edit only the WHICH line.

In [ ]:
# ===== RUN v4: improved experiments for the paper. ONE cell. =====
# Needs: Settings > Accelerator = GPU T4 x2 (or P100), Settings > Internet = On.
# Best used with: Save Version > Save & Run All (Commit), so it keeps running if the browser closes. (Max session about 9-12 h.)
# Edit ONLY the WHICH line. With 24 GPU hours, use THREE separate Save & Run All sessions (a Kaggle session lasts at most about 12 h):
#   session 1: WHICH = ["A", "B"]     (main run + second judge seed, about 6-8 h)
#   session 2: WHICH = ["C", "D"]     (third judge seed + 3B judge, about 3-4 h; the re-ranker score cache is rebuilt, about 1 h)
#   ADAPTIVE ON MORE SEEDS (the paper's main open caveat): WHICH = ["H", "I"]  (adaptive estimator only, seeds 1 and 2; about 2 h for H incl. cache rebuild, about 45 min more for I)
#   session 3: WHICH = ["G"]          (4B re-ranker, about 2-3 h)  -- optional
# After each session download results_all.zip. For pooled intervals over all seeds, send me the three zips (or run merge_seeds.py on the perq_*.npz files).
WHICH = ["A", "B"]
# Rough time on a T4 (first run also fills the re-ranker score cache, later runs reuse it): A about 4-6 h, B and C about 1-1.5 h each, D about 2 h, G about 2-3 h.
# If the session stops, just run the cell again: finished runs are skipped and cached scores are reused.
import base64, os, subprocess, sys, time, glob, zipfile
WD = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
os.chdir(WD)
V4 = (
    "..."
)
MS = (
    "IyBKb2ludCBib290c3RyYXAgb3ZlciBxdWVyaWVzIEFORCBqdWRnZSBzZWVkcy4gUnVuIG9uIENQVSBhZnRlciBydW5zIEEsIEIs"
    "IEMgKGFueSBtYWNoaW5lLCBubyBHUFUgbmVlZGVkKToKIyAgIHB5dGhvbiBtZXJnZV9zZWVkcy5weSBwZXJxX2oxLjVCX3MwLm5w"
    "eiBwZXJxX2oxLjVCX3MxLm5weiBwZXJxX2oxLjVCX3MyLm5wegojIEZvciBldmVyeSAocmUtcmFua2VyLCBzY2VuYXJpbykgaXQg"
    "YXZlcmFnZXMgdGhlIHBlci1xdWVyeSBjb3JyZWN0bmVzcyBvdmVyIHRoZSBzZWVkcywgdGhlbiBib290c3RyYXBzIHRoZSBRVUVS"
    "SUVTCiMgKHNhbWUgcmVzYW1wbGVkIHF1ZXJpZXMgZm9yIGV2ZXJ5IHNlZWQpLCBzbyB0aGUgaW50ZXJ2YWwgcmVmbGVjdHMgdGVz"
    "dC1zZXQgc2l6ZSwgYW5kIGl0IGFsc28gcmVwb3J0cyB0aGUgc3ByZWFkIG92ZXIgc2VlZHMuCmltcG9ydCBzeXMsIG51bXB5IGFz"
    "IG5wLCBwYW5kYXMgYXMgcGQKZmlsZXMgPSBzeXMuYXJndlsxOl07IGFzc2VydCBsZW4oZmlsZXMpID49IDIsICJnaXZlIGF0IGxl"
    "YXN0IHR3byBwZXJxXyoubnB6IGZpbGVzIgpaID0gW25wLmxvYWQoZiwgYWxsb3dfcGlja2xlPVRydWUpIGZvciBmIGluIGZpbGVz"
    "XQp0MCA9IFpbMF1bIl90ZXN0X3RpdGxlcyJdCmZvciB6IGluIFpbMTpdOiBhc3NlcnQgKHpbIl90ZXN0X3RpdGxlcyJdID09IHQw"
    "KS5hbGwoKSwgImRpZmZlcmVudCB0ZXN0IHNldHM6IHJ1bnMgbXVzdCB1c2UgdGhlIHNhbWUgTl9RVUVSSUVTIC8gSE9MRDEwIC8g"
    "Tl9ESVNUUiAvIFBBU1NfU0VUIgprZXlzID0gc29ydGVkKGsgZm9yIGsgaW4gWlswXS5maWxlcyBpZiBub3Qgay5zdGFydHN3aXRo"
    "KCJfIikgYW5kIGFsbChrIGluIHouZmlsZXMgZm9yIHogaW4gWikpCnJuZyA9IG5wLnJhbmRvbS5kZWZhdWx0X3JuZygwKTsgbiA9"
    "IGxlbih0MCk7IEIgPSA1MDAwOyBpZHggPSBybmcuaW50ZWdlcnMoMCwgbiwgKEIsIG4pKQptZWFuX3ZlYyA9IHtrOiBucC5tZWFu"
    "KFt6W2tdIGZvciB6IGluIFpdLCAwKSBmb3IgayBpbiBrZXlzfQpzZWVkX21lYW5zID0ge2s6IG5wLmFycmF5KFt6W2tdLm1lYW4o"
    "KSBmb3IgeiBpbiBaXSkgZm9yIGsgaW4ga2V5c30KZGVmIGNpKHYpOiBtID0gdltpZHhdLm1lYW4oMSk7IHJldHVybiAxMDAgKiB2"
    "Lm1lYW4oKSwgMTAwICogbnAucGVyY2VudGlsZShtLCAyLjUpLCAxMDAgKiBucC5wZXJjZW50aWxlKG0sIDk3LjUpCnJvd3MgPSBb"
    "XTsgZ3JvdXBzID0gc29ydGVkKHt0dXBsZShrLnNwbGl0KCJAIilbOjJdKSBmb3IgayBpbiBrZXlzfSkKcmVmcyA9IFsibWV0YWRh"
    "dGFfb25seSIsICJtZXRhZGF0YV90aHJlc2giLCAiZGVjYXkiLCAiY29udGVudF9mdCIsICJub2dhdGVfZnQiLCAiZ2F0ZWRfenMi"
    "XQpmb3Igcm4sIHNjIGluIGdyb3VwczoKICAgIGcgPSBmIntybn1Ae3NjfUBnYXRlZF9mdCIKICAgIGZvciBmYW0gaW4gWyJnYXRl"
    "ZF9mdCJdICsgcmVmcyArIFsiZGF0ZV9mcmVlIl06CiAgICAgICAgayA9IGYie3JufUB7c2N9QHtmYW19IgogICAgICAgIGlmIGsg"
    "bm90IGluIG1lYW5fdmVjOiBjb250aW51ZQogICAgICAgIG0sIGxvLCBoaSA9IGNpKG1lYW5fdmVjW2tdKTsgcm93ID0gZGljdChy"
    "ZXJhbmtlcj1ybiwgc2NlbmFyaW89c2MsIGZhbWlseT1mYW0sIG1lYW49cm91bmQobSwgMSksIGNpPWYiW3tsbzouMWZ9LHtoaTou"
    "MWZ9XSIsIHNlZWRfc2Q9cm91bmQoMTAwICogc2VlZF9tZWFuc1trXS5zdGQoZGRvZj0xKSwgMiksIHNlZWRfbWluPXJvdW5kKDEw"
    "MCAqIHNlZWRfbWVhbnNba10ubWluKCksIDEpLCBzZWVkX21heD1yb3VuZCgxMDAgKiBzZWVkX21lYW5zW2tdLm1heCgpLCAxKSkK"
    "ICAgICAgICBpZiBmYW0gIT0gImdhdGVkX2Z0IiBhbmQgZyBpbiBtZWFuX3ZlYzoKICAgICAgICAgICAgZCwgZGxvLCBkaGkgPSBj"
    "aShtZWFuX3ZlY1tnXSAtIG1lYW5fdmVjW2tdKTsgcm93WyJnYXRlZF9mdF9taW51c190aGlzIl0gPSBmIntkOisuMWZ9IFt7ZGxv"
    "OisuMWZ9LHtkaGk6Ky4xZn1dIgogICAgICAgIHJvd3MuYXBwZW5kKHJvdykKZGYgPSBwZC5EYXRhRnJhbWUocm93cyk7IGRmLnRv"
    "X2NzdigicG9vbGVkX3NlZWRfaW50ZXJ2YWxzLmNzdiIsIGluZGV4PUZhbHNlKTsgcHJpbnQoZGYudG9fc3RyaW5nKGluZGV4PUZh"
    "bHNlKSk7IHByaW50KCJzYXZlZCBwb29sZWRfc2VlZF9pbnRlcnZhbHMuY3N2IikK"
)
open("v4.py", "w").write(base64.b64decode(V4).decode("ascii"))
open("merge_seeds.py", "w").write(base64.b64decode(MS).decode("ascii"))
M15, M3 = "Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen2.5-3B-Instruct"
ALL_RUNS = {
    "A": dict(JSEED=0, JUDGE_MODEL=M15, GEN_MODEL=M3, RUN_EXTRAS=1, RUN_DIAG=1, RUN_E2E_VARIANTS=1),     # main run: all new analyses + end-to-end with two generators
    "B": dict(JSEED=1, JUDGE_MODEL=M15, RUN_EXTRAS=0, RUN_DIAG=0, RUN_E2E_VARIANTS=0),                   # second judge seed
    "C": dict(JSEED=2, JUDGE_MODEL=M15, RUN_EXTRAS=0, RUN_DIAG=0, RUN_E2E_VARIANTS=0),                   # third judge seed
    "D": dict(JSEED=0, JUDGE_MODEL=M3, RUN_EXTRAS=0, RUN_DIAG=1, RUN_E2E_VARIANTS=0),                    # larger judge
    "H": dict(JSEED=1, JUDGE_MODEL=M15, RUN_EXTRAS=1, RUN_ADAPT_ONLY=1, RUN_DIAG=0, RUN_E2E_VARIANTS=0),       # adaptive date-trust estimator only, judge seed 1
    "I": dict(JSEED=2, JUDGE_MODEL=M15, RUN_EXTRAS=1, RUN_ADAPT_ONLY=1, RUN_DIAG=0, RUN_E2E_VARIANTS=0),       # adaptive date-trust estimator only, judge seed 2
    "G": dict(JSEED=0, JUDGE_MODEL=M15, RERANKERS_SEL="qwen3-4B", N_QUERIES=1200, PASS_SET="core", RUN_EXTRAS=0, RUN_DIAG=0, RUN_E2E_VARIANTS=0),   # larger re-ranker
}
status = {}
for name in WHICH:
    cfg = ALL_RUNS[name]; done = f"run_{name}.done"
    if os.path.exists(done):
        print(f"##### run {name}: already finished, skipping"); status[name] = "done (earlier)"; continue
    print(f"\n##### run {name} starting: {cfg}"); t0 = time.time()
    env = dict(os.environ, **{k: str(v) for k, v in cfg.items()})
    with open(f"log_run_{name}.txt", "w") as lg:
        p = subprocess.Popen([sys.executable, "-u", "v4.py"], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=WD)
        for line in p.stdout:
            lg.write(line)
            if not line.startswith("update ") or "/375" in line: print(line, end="")
        p.wait()
    ok = p.returncode == 0
    status[name] = ("OK" if ok else f"FAILED (code {p.returncode}) - see log_run_{name}.txt") + f" in {(time.time()-t0)/60:.0f} min"
    if ok: open(done, "w").write("ok")
    print(f"##### run {name}: {status[name]}")
print("\n" + "=" * 20, "STATUS", "=" * 20); [print(k, v) for k, v in status.items()]
seed_files = sorted(glob.glob("perq_j1.5B_s[0-9].npz"))
if len(seed_files) >= 2:
    print("\nPooled intervals over queries and judge seeds:", seed_files)
    subprocess.run([sys.executable, "merge_seeds.py"] + seed_files)
with zipfile.ZipFile("results_all.zip", "w") as z:
    for f in glob.glob("exp4_*.csv") + glob.glob("e2e_*.csv") + glob.glob("e2e_*.txt") + glob.glob("SUMMARY_*.txt") + glob.glob("log_run_*.txt") + glob.glob("perq_*.npz") + glob.glob("pooled_seed_intervals.csv") + glob.glob("pools_*.json.gz"): z.write(f)
print("\nALL DONE. Download results_all.zip (Output tab) and also upload the SUMMARY_*.txt files to the chat.")
